# 01. Data Understanding, Cleansing & Feature Engineering
## Project: AI-Driven Loan Recovery & Risk Analytics

### Overview
This notebook establishes the foundation of the analytics workflow:
1. **Data Ingestion & Schema Inspection**: Loading 4 core banking relational tables (Customers, Loans, Repayments, Recovery Activities).
2. **Data Quality Audit**: Diagnosing missing data patterns, abnormal outliers, type mismatches, and duplicate records.
3. **Imputation & Treatment**: Applying domain-appropriate median imputations for employment length, mode imputations for housing status, and IQR-based clipping for income anomalies.
4. **Relational Merging & Master Dataset Creation**: Creating a unified analytical record for each borrower.
5. **Advanced Feature Engineering**: Deriving critical credit metrics including Debt-to-Income (DTI), Loan-to-Value (LTV), Delinquency Severity Scores, Net Recovery Amount, and Credit Risk Tiers.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
print("Environment initialized. Loading raw relational datasets...")

customers_df = pd.read_csv("../data/raw/customers.csv")
loans_df = pd.read_csv("../data/raw/loans.csv")
repayments_df = pd.read_csv("../data/raw/repayments.csv")
recovery_df = pd.read_csv("../data/raw/recovery_activities.csv")

print(f"Customers: {customers_df.shape}")
print(f"Loans: {loans_df.shape}")
print(f"Repayments: {repayments_df.shape}")
print(f"Recovery Activities: {recovery_df.shape}")


### 1. Inspecting Missing Values and Data Anomalies

In [ ]:
# Check missing values across datasets
print("--- Missing Values in Customers ---")
print(customers_df.isnull().sum()[customers_df.isnull().sum() > 0])

print("\n--- Missing Values in Loans ---")
print(loans_df.isnull().sum()[loans_df.isnull().sum() > 0])

print("\n--- Missing Values in Repayments ---")
print(repayments_df.isnull().sum()[repayments_df.isnull().sum() > 0])

print("\n--- Missing Values in Recovery Activities ---")
print(recovery_df.isnull().sum()[recovery_df.isnull().sum() > 0])


### 2. Data Cleaning & Imputation Pipeline

In [ ]:
# 1. Impute employment length with median per employment status
emp_median = customers_df.groupby("employment_status")["employment_length_years"].transform("median")
customers_df["employment_length_years"] = customers_df["employment_length_years"].fillna(emp_median).fillna(3).astype(int)

# 2. Impute housing status with mode
mode_housing = customers_df["housing_status"].mode()[0]
customers_df["housing_status"] = customers_df["housing_status"].fillna(mode_housing)

# 3. Outlier handling for annual_income
def cap_outliers(group):
    q25 = group["annual_income"].quantile(0.25)
    q75 = group["annual_income"].quantile(0.75)
    iqr = q75 - q25
    upper = q75 + 3.0 * iqr
    lower = max(12000, q25 - 1.5 * iqr)
    group["annual_income"] = np.clip(group["annual_income"], lower, upper)
    return group

customers_df = customers_df.groupby("city_tier", group_keys=False).apply(cap_outliers)

# 4. Clean loans collateral values
loans_df["collateral_value"] = loans_df["collateral_value"].fillna(0.0)
loans_df["disbursement_date"] = pd.to_datetime(loans_df["disbursement_date"])

print("Data cleansing and imputation successfully executed.")


### 3. Master Relational Merging & Feature Engineering

In [ ]:
# Merge Loans and Repayments
loans_rep_df = pd.merge(loans_df, repayments_df, on=["loan_id", "customer_id"], how="inner")
loans_rep_df["outstanding_principal"] = np.maximum(0.0, loans_rep_df["loan_amount"] - loans_rep_df["principal_paid"])

# Merge with Customers
master_df = pd.merge(customers_df, loans_rep_df, on="customer_id", how="inner")

# Left merge with Recovery Activities
master_df = pd.merge(master_df, recovery_df.drop(columns=["customer_id"]), on="loan_id", how="left")

# Fill recovery defaults for current/healthy accounts
master_df["recovery_status"] = master_df["recovery_status"].fillna("No Delinquency / Not Applicable")
master_df["primary_channel"] = master_df["primary_channel"].fillna("None")
master_df["recovered_amount"] = master_df["recovered_amount"].fillna(0.0)
master_df["recovery_cost"] = master_df["recovery_cost"].fillna(0.0)
master_df["settlement_discount_pct"] = master_df["settlement_discount_pct"].fillna(0.0)

# Engineer Financial Metrics
r = (master_df["interest_rate"] / 100.0) / 12.0
n = master_df["loan_term_months"]
master_df["monthly_emi"] = (master_df["loan_amount"] * (r * (1 + r)**n) / ((1 + r)**n - 1)).round(2)
master_df["dti_ratio"] = ((master_df["monthly_emi"] * 12) / master_df["annual_income"]).round(4)
master_df["is_secured"] = np.where(master_df["collateral_type"] != "None", 1, 0)
master_df["ltv_ratio"] = np.where(master_df["is_secured"] == 1, np.clip(master_df["loan_amount"] / np.maximum(master_df["collateral_value"], 1.0), 0.1, 1.5), 1.0).round(4)

# Credit Risk Tiers
bins = [0, 549, 649, 699, 749, 900]
labels = ["Deep Subprime (<550)", "Subprime (550-649)", "Near-Prime (650-699)", "Prime (700-749)", "Super-Prime (750+)"]
master_df["credit_risk_tier"] = pd.cut(master_df["credit_score"], bins=bins, labels=labels)

master_df["net_recovery_amount"] = (master_df["recovered_amount"] - master_df["recovery_cost"]).round(2)

print(f"Master Clean Dataset Shape: {master_df.shape}")
display(master_df.head(3))


### 4. Summary Statistics & Data Export

In [ ]:
print("=== Master Dataset Numerical Summary ===")
display(master_df[["annual_income", "credit_score", "loan_amount", "interest_rate", "dti_ratio", "overdue_days", "recovered_amount"]].describe().T)
